# 심화 미션: 배터리 셀 검사 기록
- 상황: 출하 전 검사에서 불합격 셀을 미리 골라내고 싶다
- 목표: 오늘 배운 순서를 다른 데이터로 혼자 한 바퀴 돌린다

### 용어 풀이 - 배터리 검사에서 쓰는 말

| 말 | 뜻 |
|---|---|
| 합격 / 불합격 | 검사에서 붙이는 판정 |
| 개방전압 (open_voltage_V) | 부하 없이 잰 셀의 전압 |
| 용량 (capacity_mAh) | 완충했을 때 담을 수 있는 전기량 |
| 내부저항 (internal_resistance_mOhm) | 전류가 흐를 때 셀 내부에서 걸리는 저항 |
| 충전 시간 (charge_time_min) | 완충까지 걸린 시간 |
| 챔버 온도 / 습도 | 검사할 때 챔버 안 환경 |

## Q1. 파일 열고 크기 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/day04_battery.csv")

print("행 수, 열 수:", df.shape)
df.head()

행 수, 열 수: (2847, 14)


,cell_id,inspected_at,line,shift,open_voltage_V,capacity_mAh,internal_resistance_mOhm,thickness_mm,weight_g,charge_time_min,chamber_temp_C,chamber_humidity_pct,inspector,result
0,CELL-00001,2026-03-02 09:01,B라인,주간,3.679,4957.2,20.71,11.072,69.53,73.5,27.1,41.1,한서린,합격
1,CELL-00002,2026-03-02 09:52,B라인,야간,3.662,4821.1,18.68,10.876,72.42,80.1,24.7,46.3,박정민,합격
2,CELL-00003,2026-03-02 10:35,B라인,주간,3.585,4769.6,21.58,11.126,70.21,74.3,23.3,40.1,한서린,합격
3,CELL-00004,2026-03-02 12:08,B라인,주간,3.719,5092.1,21.24,10.862,70.85,57.0,26.3,46.6,한서린,합격
4,CELL-00005,2026-03-02 12:21,B라인,주간,3.638,4814.8,15.06,10.897,69.72,63.0,24.0,52.0,박정민,합격


배터리 셀 최종검사 기록 (2026년 3~6월)

| 열 | 뜻 |
| --- | --- |
| `cell_id` | 셀 고유 번호 |
| `inspected_at` | 검사한 날짜와 시각 |
| `line` | 조립 라인 (A라인 / B라인 / C라인) |
| `shift` | 검사한 조 (주간 / 야간) |
| `open_voltage_V` | 개방 전압 (V) |
| `capacity_mAh` | 용량 (mAh) |
| `internal_resistance_mOhm` | 내부 저항 (mΩ) |
| `thickness_mm` | 두께 (mm) |
| `weight_g` | 무게 (g) |
| `charge_time_min` | 완충까지 걸린 시간 (분) |
| `chamber_temp_C` | 검사실 온도 (℃) |
| `chamber_humidity_pct` | 검사실 습도 (%) |
| `inspector` | 검사한 사람 |
| `result` | **합격 / 불합격** — 맞혀야 할 정답 |

> ⚠️ **숫자가 아닌 열이 섞여 있습니다.** 셀 번호·검사 시각·라인·조·검사원은 숫자가 아니에요. 입력으로 넣을 열을 고를 때 주의하세요.
>

## Q2. 불합격은 얼마나 드문가

In [2]:
print(df["result"].value_counts())
print(df["result"].value_counts(normalize=True) * 100)

result
합격     2714
불합격     133
Name: count, dtype: int64
result
합격     95.328416
불합격     4.671584
Name: proportion, dtype: float64


## Q3. 남은 빈칸 채우기

In [3]:
feature_cols = [
    "open_voltage_V", "capacity_mAh", "internal_resistance_mOhm",
    "thickness_mm", "weight_g", "charge_time_min",
    "chamber_temp_C", "chamber_humidity_pct",
]

print("채우기 전")
for col in feature_cols:
    빈칸수 = df[col].isna().sum()
    if 빈칸수 > 0:
        print(f"  {col}: {빈칸수}개")

# 열마다 그 열의 중앙값으로 빈칸을 채운다
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

print(f"채운 뒤 남은 빈칸: {df[feature_cols].isna().sum().sum()}개")

채우기 전
  charge_time_min: 38개
  chamber_temp_C: 57개
  chamber_humidity_pct: 113개
채운 뒤 남은 빈칸: 0개


## Q4. 입력과 정답 가르기

In [4]:
# result가 "불합격"이면 1, 아니면 0인 숫자 열을 새로 만든다
df["불합격여부"] = (df["result"] == "불합격").astype(int)

X = df[feature_cols]
y = df["불합격여부"]

print("입력 크기:", X.shape)
print(f"정답 분포: 0이 {(y == 0).sum()}건, 1이 {(y == 1).sum()}건")

입력 크기: (2847, 8)
정답 분포: 0이 2714건, 1이 133건


## Q5. 학습용과 시험용으로 나누기

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"학습용 {len(X_train)}건 (불합격 {(y_train == 1).sum()}건, {round((y_train == 1).mean() * 100, 2)}%)")
print(f"시험용 {len(X_test)}건 (불합격 {(y_test == 1).sum()}건, {round((y_test == 1).mean() * 100, 2)}%)")

학습용 2277건 (불합격 106건, 4.66%)
시험용 570건 (불합격 27건, 4.74%)


## Q6. 아무것도 배우지 않은 기준 모델

In [6]:
import numpy as np

기준예측 = np.zeros(len(y_test), dtype=int)
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")
print("불합격이라 지목한 건수:", (기준예측 == 1).sum(), "건")

기준 모델 정확도: 95.26 %
불합격이라 지목한 건수: 0 건


### 결과 정리 (실행 결과 기준)

| 정확도 | 잡은 불합격 | 놓친 불합격 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|
| 95.26% | 0 | 27 | 0 | 0.0 | 0.0 | 0.0 |

## Q7. 손대지 않은 모델로 한 번 해보기

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, f1_score

# 단위를 맞추기 위한 표준화
scaler = StandardScaler()
X_train_스케일 = scaler.fit_transform(X_train)
X_test_스케일 = scaler.transform(X_test)

# 설정을 아무것도 건드리지 않은 로지스틱 회귀
model = LogisticRegression()
model.fit(X_train_스케일, y_train)
예측 = model.predict(X_test_스케일)

맞힌합격, 헛경보, 놓친불합격, 잡은불합격 = confusion_matrix(y_test, 예측).ravel()

print("정확도:", round(accuracy_score(y_test, 예측) * 100, 2), "%")
print("잡은 불합격:", 잡은불합격, "/ 놓친 불합격:", 놓친불합격, "/ 헛경보:", 헛경보)
print("재현율:", round(recall_score(y_test, 예측), 3),
      " 정밀도:", round(precision_score(y_test, 예측, zero_division=0), 3),
      " F1:", round(f1_score(y_test, 예측), 3))

정확도: 97.02 %
잡은 불합격: 12 / 놓친 불합격: 15 / 헛경보: 2
재현율: 0.444  정밀도: 0.857  F1: 0.585


### 결과 정리 (실행 결과 기준)

| 정확도 | 잡은 불합격 | 놓친 불합격 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|
| 97.02% | 12 | 15 | 2 | 0.444 | 0.857 | 0.585 |

## Q8. 드문 쪽에 무게 주고 다시 하기

In [8]:
# class_weight="balanced" - 드문 쪽(불합격) 한 건을 더 무겁게 세도록 알려준다. 나머지는 Q7과 같다
가중치모델 = LogisticRegression(max_iter=1000, class_weight="balanced")
가중치모델.fit(X_train_스케일, y_train)
가중치예측 = 가중치모델.predict(X_test_스케일)

# Q7("손 안 댐")의 예측(예측)은 그대로 두고, 비교를 위해 지표만 다시 잰다
맞힌합격7, 헛경보7, 놓친불합격7, 잡은불합격7 = confusion_matrix(y_test, 예측).ravel()
맞힌합격8, 헛경보8, 놓친불합격8, 잡은불합격8 = confusion_matrix(y_test, 가중치예측).ravel()

print(f"{'처리':6} {'정확도':>8} {'잡은 불합격':>10} {'놓친 불합격':>10} {'헛경보':>6} {'재현율':>8} {'정밀도':>8} {'F1':>8}")
print(f"{'손 안 댐':6} {round(accuracy_score(y_test, 예측)*100,2):>7}% {잡은불합격7:>10} {놓친불합격7:>10} {헛경보7:>6} {round(recall_score(y_test, 예측),3):>8} {round(precision_score(y_test, 예측, zero_division=0),3):>8} {round(f1_score(y_test, 예측),3):>8}")
print(f"{'가중치':6} {round(accuracy_score(y_test, 가중치예측)*100,2):>7}% {잡은불합격8:>10} {놓친불합격8:>10} {헛경보8:>6} {round(recall_score(y_test, 가중치예측),3):>8} {round(precision_score(y_test, 가중치예측, zero_division=0),3):>8} {round(f1_score(y_test, 가중치예측),3):>8}")

처리          정확도     잡은 불합격     놓친 불합격    헛경보      재현율      정밀도       F1
손 안 댐    97.02%         12         15      2    0.444    0.857    0.585
가중치      86.32%         25          2     76    0.926    0.248    0.391


### 결과 정리 (실행 결과 기준)

| 처리 | 정확도 | 잡은 불합격 | 놓친 불합격 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|---|
| 손 안 댐 | 97.02% | 12 | 15 | 2 | 0.444 | 0.857 | 0.585 |
| 가중치 | 86.32% | 25 | 2 | 76 | 0.926 | 0.248 | 0.391 |

## Q9. 의사결정나무 - 깊이 바꿔가며 해보기

In [9]:
from sklearn.tree import DecisionTreeClassifier

# class_weight="balanced" - 드문 쪽(불합격)을 무겁게 센다. 나무의 고정 번호는 42
print(f"{'깊이':6} {'정확도':>8} {'잡은 불합격':>10} {'헛경보':>6} {'재현율':>8} {'F1':>8}")
for 깊이 in [3, 5, None]:
    나무 = DecisionTreeClassifier(class_weight="balanced", random_state=42, max_depth=깊이)
    나무.fit(X_train, y_train)
    나무예측 = 나무.predict(X_test)

    맞힌합격, 헛경보, 놓친불합격, 잡은불합격 = confusion_matrix(y_test, 나무예측).ravel()
    정확도 = accuracy_score(y_test, 나무예측)
    재현율 = recall_score(y_test, 나무예측, zero_division=0)
    f1 = f1_score(y_test, 나무예측, zero_division=0)

    깊이표시 = 깊이 if 깊이 is not None else "제한 없음"
    print(f"{str(깊이표시):6} {round(정확도*100,2):>7}% {잡은불합격:>10} {헛경보:>6} {round(재현율,3):>8} {round(f1,3):>8}")

깊이          정확도     잡은 불합격    헛경보      재현율       F1
3        88.42%         24     63    0.889    0.421
5        85.79%         19     73    0.704    0.319
제한 없음    92.98%         11     24    0.407    0.355


### 결과 정리 (실행 결과 기준)

| 깊이 | 정확도 | 잡은 불합격 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 3 | 88.42% | 24 | 63 | 0.889 | 0.421 |
| 5 | 85.79% | 19 | 73 | 0.704 | 0.319 |
| 제한 없음 | 92.98% | 11 | 24 | 0.407 | 0.355 |

Q9. 다이얼 세 번 돌리기

## Q10. 설정값 자동 탐색하기

In [10]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

설정후보 = {
    "max_depth": [2, 3, 4, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
}

# 다섯 겹으로 나누되 섞어서 나눈다 (겹 나누기도 고정 번호 42)
겹나누기 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# class_weight="balanced" - 드문 쪽(불합격)을 무겁게 센다. 나무의 고정 번호는 42
탐색 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=설정후보,
    scoring="f1",
    cv=겹나누기,
)
# 학습용만 써서 탐색한다
탐색.fit(X_train, y_train)

깊이표시 = 탐색.best_params_["max_depth"] if 탐색.best_params_["max_depth"] is not None else "제한 없음"
print(f"1등 설정: 깊이 {깊이표시}, 끝자리 최소 {탐색.best_params_['min_samples_leaf']}")
print("탐색 점수(F1):", round(탐색.best_score_, 3))
print()

최고모델 = 탐색.best_estimator_
최고예측 = 최고모델.predict(X_test)

맞힌합격, 헛경보, 놓친불합격, 잡은불합격 = confusion_matrix(y_test, 최고예측).ravel()

print("시험용 채점")
print(f"  정확도 {round(accuracy_score(y_test, 최고예측) * 100, 1)}%  잡은 불합격 {잡은불합격}  헛경보 {헛경보}")
print(f"  재현율 {round(recall_score(y_test, 최고예측, zero_division=0), 3)}",
      f" 정밀도 {round(precision_score(y_test, 최고예측, zero_division=0), 3)}",
      f" F1 {round(f1_score(y_test, 최고예측, zero_division=0), 3)}")

1등 설정: 깊이 10, 끝자리 최소 1
탐색 점수(F1): 0.38

시험용 채점
  정확도 91.4%  잡은 불합격 15  헛경보 37
  재현율 0.556  정밀도 0.288  F1 0.38


### 결과 정리 (실행 결과 기준)

| 설정 | 탐색 점수(F1) | 정확도 | 잡은 불합격 | 헛경보 | 재현율 | 정밀도 | F1 |
|---|---|---|---|---|---|---|---|
| 깊이 10, 끝자리 최소 1 | 0.38 | 91.4% | 15 | 37 | 0.556 | 0.288 | 0.38 |

**인사이트**: 학습용 교차검증 점수(F1 0.38)와 시험용 F1(0.38)이 거의 같아 믿을 만하지만, Q9에서 직접 시도했던 깊이 3(F1 0.421)보다는 낮다 — 자동 탐색이 항상 가장 좋은 설정을 찾아주는 것은 아니다.

## Q11. 세 모델을 교차검증으로 나란히 재기

In [11]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

# 겹마다 불합격 비율을 맞추고, 섞어서 나누되 고정 번호는 42
겹나누기 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

모델들 = {
    "손 안 댐": make_pipeline(StandardScaler(), LogisticRegression()),
    "가중치": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "나무(탐색 1등)": DecisionTreeClassifier(random_state=42, class_weight="balanced",
                                          max_depth=탐색.best_params_["max_depth"],
                                          min_samples_leaf=탐색.best_params_["min_samples_leaf"]),
}

print(f"{'모델':14} {'재현율 평균':>10} {'흔들림':>8} {'F1 평균':>10} {'흔들림':>8}")
for 이름, 모델 in 모델들.items():
    # 학습용(X_train, y_train)만 넣는다
    결과 = cross_validate(모델, X_train, y_train, cv=겹나누기, scoring=["recall", "f1"])
    재현율평균 = 결과["test_recall"].mean()
    재현율흔들림 = 결과["test_recall"].std()
    f1평균 = 결과["test_f1"].mean()
    f1흔들림 = 결과["test_f1"].std()
    print(f"{이름:14} {round(재현율평균,3):>10} {round(재현율흔들림,3):>8} {round(f1평균,3):>10} {round(f1흔들림,3):>8}")

모델                 재현율 평균      흔들림      F1 평균      흔들림
손 안 댐               0.406    0.064      0.517    0.085
가중치                 0.859    0.064      0.385    0.029
나무(탐색 1등)             0.5    0.067       0.38     0.05


### 결과 정리 (실행 결과 기준)

| 모델 | 재현율 평균 | 흔들림 | F1 평균 | 흔들림 |
|---|---|---|---|---|
| 손 안 댐 | 0.406 | 0.064 | 0.517 | 0.085 |
| 가중치 | 0.859 | 0.064 | 0.385 | 0.029 |
| 나무(탐색 1등) | 0.500 | 0.067 | 0.380 | 0.050 |

**한 줄 인사이트**: 재현율만 보면 가중치 로지스틱이 압도적이지만 F1은 오히려 손 안 댄 모델이 가장 높다 — 무엇을 1등 기준으로 삼느냐에 따라 추천 모델이 뒤바뀐다.

## Q12. 담당자께 드리는 권고

- 불합격을 놓치는 비용이 헛경보 비용보다 훨씬 크다면 **가중치 준 로지스틱 회귀**를 추천합니다. 재현율 평균 0.859로 불합격 열 건 중 여덟~아홉 건을 잡아냅니다.
- 다만 그만큼 헛경보(정상인데 불합격으로 잘못 지목)도 늘어나므로, 검사 인력이 재검사를 감당할 수 있는지 먼저 확인해야 합니다.
- 재검사 여력이 부족하다면 F1이 가장 높은 **손대지 않은 로지스틱 회귀**로 시작해 점차 가중치를 올려보는 절충안을 권합니다.